In [1]:
import utils
import keras
import numpy as np
from aim.keras import AimCallback
import pandas as pd

2026-03-06 14:03:35.030920: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 14:03:35.987460: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 14:03:46.856486: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
columns = ("gate1", "gate2", "lighting")
(train_x, train_y), (val_x, val_y), (test_x, test_y) = utils.read_stratified_data(columns=columns)

/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/20260303_182354203_iOS.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/275815406_Andrew-Crowley-1_trans_NvBQzQNjv4BqJgZjG4XE8BZGTSy9SLp5TPzOL_bEerq8T.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/kisspng-2018-tesla-model-s-tesla-motors-car-ele_fLj30kA.2230115115224762368998.png
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/20260303_182059663_iOS.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/black-tesla-model-x-carbon-fiber-spoiler-mx22-forged-aftermarket-wheel_tCrLNla.jpg
/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/datasett/Edmu_EmY1CYe.-2020-Tesla-Model-Ywire-38540856-1611748943-973_634x3962-1280x720.jpg


In [3]:
print(train_x.shape)
print(train_y.shape)
print(val_x.shape)
print(val_y.shape)
print(test_x.shape)
print(test_y.shape)

(3768, 300, 300, 3)
(3768, 6)
(805, 300, 300, 3)
(805, 6)
(807, 300, 300, 3)
(807, 6)


In [4]:
input = keras.Input(shape=(train_x[0].shape))

# x = keras.layers.Conv2D(filters=32, kernel_size=3, activation="relu")(input)
# x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu")(input)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Flatten()(x)

gate1 = keras.layers.Dense(1, activation="sigmoid", name="gate1")(x)
gate2 = keras.layers.Dense(8, activation="softmax", name="gate2")(x)

model = keras.Model(inputs=input, outputs={"gate1": gate1, "gate2": gate2}, name="Overfitted_model")
model.summary()

opt = keras.optimizers.Adam(learning_rate=6e-4)

model.compile(
    optimizer=opt,
    loss={
        "gate1": keras.losses.BinaryCrossentropy(),
        "gate2": keras.losses.SparseCategoricalCrossentropy(),
    },
    metrics={
        "gate1": [keras.metrics.BinaryAccuracy(name="acc")],
    },
    weighted_metrics={
        "gate2": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    },
)


I0000 00:00:1772802295.387253   13535 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9569 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "Overfitted_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 298, 298,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 149, 149,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 73, 73,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 682112)    │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate1 (Dense)       │ (None, 1)         │    682,113 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gate2 (Dense)       │ (None, 8)         │  5,456,904 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,214,665 (23.71 MB)

 Trainable params: 6,214,665 (23.71 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# In case of any issues the following command can be used to restore data:
# aim storage --repo /mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions restore '*'


batch_size = 16
epochs = 10
aim_cb = AimCallback(
    repo=".",                      
    experiment="Vehicle_Gate_Test",
)

hparams = {
    "learning_rate": opt.learning_rate,
    "batch_size": batch_size,
    "epochs": epochs,
    "optimizer": "Adam",
    "model": "Overfitted_model",
}

sample_weight={
    "gate1": np.ones(len(train_y)),
    "gate2": 1 - train_y['gate1'] 
}

validation_data=(val_x, {'gate1': val_y["gate1"], 'gate2': val_y["gate2"]})

y = {'gate1': train_y["gate1"], 'gate2': train_y["gate2"]}
history = model.fit(epochs=epochs, batch_size=batch_size, x=train_x, y=y, sample_weight=sample_weight, validation_data=validation_data, callbacks=[aim_cb])

Epoch 1/10


2026-03-06 14:05:06.906962: I external/local_xla/xla/service/service.cc:163] XLA service 0x6388923bd880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-06 14:05:06.906996: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3080, Compute Capability 8.6
2026-03-06 14:05:07.085054: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-06 14:05:07.557627: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-03-06 14:05:08.280623: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 8 bytes spill stores, 8 bytes spill loads

2026-03-06 14:05:08.282955: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warni

  5/236 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - gate1_acc: 0.3783 - gate1_loss: 2.5750 - gate2_acc: 0.2052 - gate2_loss: 2.2387 - loss: 4.8137   

I0000 00:00:1772802313.488217   14000 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


235/236 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - gate1_acc: 0.6001 - gate1_loss: 0.8380 - gate2_acc: 0.3469 - gate2_loss: 0.9497 - loss: 1.7877

2026-03-06 14:05:20.565660: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.30GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


236/236 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - gate1_acc: 0.6004 - gate1_loss: 0.8372 - gate2_acc: 0.3470 - gate2_loss: 0.9490 - loss: 1.7863

2026-03-06 14:05:25.023838: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.55GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-03-06 14:05:26.361715: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.20GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


236/236 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - gate1_acc: 0.6608 - gate1_loss: 0.6563 - gate2_acc: 0.3571 - gate2_loss: 0.7862 - loss: 1.4429 - val_gate1_acc: 0.7031 - val_gate1_loss: 0.5862 - val_gate2_acc: 0.1826 - val_gate2_loss: 1.8420 - val_loss: 2.4276
Epoch 2/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - gate1_acc: 0.7800 - gate1_loss: 0.4656 - gate2_acc: 0.6610 - gate2_loss: 0.4011 - loss: 0.8664 - val_gate1_acc: 0.7130 - val_gate1_loss: 0.5555 - val_gate2_acc: 0.1689 - val_gate2_loss: 2.8689 - val_loss: 3.4081
Epoch 3/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - gate1_acc: 0.9180 - gate1_loss: 0.2114 - gate2_acc: 0.9700 - gate2_loss: 0.0570 - loss: 0.2685 - val_gate1_acc: 0.7354 - val_gate1_loss: 0.6113 - val_gate2_acc: 0.1764 - val_gate2_loss: 3.2953 - val_loss: 3.8958
Epoch 4/10
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - gate1_acc: 0.9843 - gate1_loss: 0.0623 - gate2_acc: 0.9969 - gate2_loss: 0.0158 - loss: 0.0783 - val_gate1_acc: 0.7652 - val_gate1_loss: 0.7638 - val_gate

In [6]:
def predict_heads(model, data_x):
    """Returns (lvl1_probs, lvl2_probs) for the overfitted model."""
    preds = model.predict(data_x, verbose=0)
    # Mapping 'gate1' -> lvl1 and 'gate2' -> lvl2
    return preds['gate1'].flatten(), preds['gate2']

# Get predictions once to use in all tables
p1, p2 = predict_heads(model, val_x)
# Extract ground truth as numpy arrays
y1_true = val_y['gate1'].to_numpy()
y2_true = val_y['gate2'].to_numpy()

In [7]:
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

def eval_lvl1_by_lighting(y_true, p1_probs, lighting_series, threshold=0.5):
    rows = []
    y_pred = (p1_probs >= threshold).astype(int)
    
    for cond in ["Light", "Medium", "Dark"]:
        mask = (lighting_series == cond)
        if not mask.any(): continue
        
        yt, yp = y_true[mask], y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        
        rows.append({
            "lighting": cond,
            "n_total": len(yt),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "TPR": tp / (tp + fn) if (tp + fn) > 0 else 0,
            "TNR": tn / (tn + fp) if (tn + fp) > 0 else 0,
            "lvl1_acc": accuracy_score(yt, yp),
            "lvl1_bal_acc": balanced_accuracy_score(yt, yp),
            "lvl1_f1": f1_score(yt, yp)
        })
    return pd.DataFrame(rows)

print("\n--- TABLE 1: LEVEL 1 BY LIGHTING ---")
display(eval_lvl1_by_lighting(y1_true, p1, val_y['lighting']))


--- TABLE 1: LEVEL 1 BY LIGHTING ---


,lighting,n_total,TN,FP,FN,TP,TPR,TNR,lvl1_acc,lvl1_bal_acc,lvl1_f1
0,Light,441,98,62,31,250,0.889680,0.612500,0.789116,0.751090,0.843170
1,Medium,183,58,32,17,76,0.817204,0.644444,0.732240,0.730824,0.756219
2,Dark,181,54,37,18,72,0.800000,0.593407,0.696133,0.696703,0.723618


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support
import pandas as pd
# 1. Get Predictions
preds = model.predict(val_x, verbose=0)
p1 = preds['gate1'].flatten() # Level 1 probabilities
p2 = preds['gate2']           # Level 2 probabilities (softmax)

# 2. Prepare Ground Truth & Predictions
y1_true = val_y['gate1'].to_numpy()
y2_true = val_y['gate2'].to_numpy().fillna(-1)
y2_pred = np.argmax(p2, axis=1)

# Mask for Tesla samples only (Level 2 is only valid when Lvl1 == 1)
tesla_mask = (y1_true == 1)
yt2_tesla = y2_true[tesla_mask]
yp2_tesla = y2_pred[tesla_mask]
lighting_tesla = val_y['lighting'].to_numpy()[tesla_mask]

# --- TABLE: LEVEL 2 OVERALL ---
def eval_lvl2_overall(yt, yp):
    return pd.DataFrame([{
        "n_tesla": len(yt),
        "lvl2_acc": accuracy_score(yt, yp),
        "lvl2_bal_acc": balanced_accuracy_score(yt, yp),
        "lvl2_f1_macro": f1_score(yt, yp, average='macro'),
        "lvl2_maj_acc": (yt == pd.Series(yt).mode()[0]).mean()
    }])

print("\n--- LEVEL 2 OVERALL ---")
display(eval_lvl2_overall(yt2_tesla, yp2_tesla))

# --- TABLE: LEVEL 2 BY LIGHTING ---
def eval_lvl2_lighting(yt, yp, light):
    rows = []
    for cond in ["Light", "Medium", "Dark"]:
        m = (light == cond)
        if not m.any(): continue
        rows.append({
            "lighting": cond,
            "n_tesla": m.sum(),
            "lvl2_acc": accuracy_score(yt[m], yp[m]),
            "lvl2_bal_acc": balanced_accuracy_score(yt[m], yp[m]),
            "lvl2_f1_macro": f1_score(yt[m], yp[m], average='macro')
        })
    return pd.DataFrame(rows)

print("\n--- LEVEL 2 BY LIGHTING ---")
display(eval_lvl2_lighting(yt2_tesla, yp2_tesla, lighting_tesla))

# --- TABLE: LEVEL 2 PER-CLASS ---
p, r, f, s = precision_recall_fscore_support(yt2_tesla, yp2_tesla, labels=range(len(TESLA_NAMES)), zero_division=0)
class_metrics = pd.DataFrame({
    'class_id': range(len(TESLA_NAMES)),
    'class': TESLA_NAMES,
    'support': s,
    'precision': p,
    'recall': r,
    'f1': f
}).sort_values('recall') # Baseline sorts by recall

print("\n--- LEVEL 2 PER-CLASS ---")
display(class_metrics)

# --- TABLE: TOP MISCLASSIFICATIONS ---
misclassed = []
for t_id in range(len(TESLA_NAMES)):
    for p_id in range(len(TESLA_NAMES)):
        if t_id == p_id: continue
        count = ((yt2_tesla == t_id) & (yp2_tesla == p_id)).sum()
        support = (yt2_tesla == t_id).sum()
        if count > 0:
            misclassed.append({
                'true': TESLA_NAMES[t_id],
                'pred': TESLA_NAMES[p_id],
                'rate': count / support,
                'support_true': support
            })

top_mis = pd.DataFrame(misclassed).sort_values('rate', ascending=False).head(5)
print("\n--- TOP MISCLASSIFICATIONS ---")
display(top_mis)


--- LEVEL 2 OVERALL ---


/mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv/lib/python3.10/site-packages/sklearn/externals/array_api_compat/numpy/_aliases.py:125: RuntimeWarning: invalid value encountered in cast
  return x.astype(dtype=dtype, copy=copy)


ValueError: Input y_true contains NaN.